# parents-dict-by-argidx — ex2: build_parents_full — extend parents dict to include kwarg Tensors keyed by name

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parents-dict-by-argidx`. Running the final beacon cell reports progress against the `Backprop: Parents dict by argidx` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parents dict by argidx` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parents-dict-by-argidx`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parents-dict-by-argidx"
DD_SUBTOPIC = "Backprop: Parents dict by argidx"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parents dict by argidx — quick refresher

`Recipe.parents` maps the arg's original index (positional) or name (kwarg) to its parent Tensor, skipping non-Tensor inputs.

**Worked exemplar.** `op(t1, 3.0, kw=t2)` → `parents == {0: t1, 'kw': t2}`. Positional float skipped; kwarg Tensor keyed by its kwarg name (string), not by a positional index.

### Exercise 2 — build_parents_full — extend parents dict to include kwarg Tensors keyed by name

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the parents-dict construction over BOTH positional args and kwargs: positional Tensors keyed by argidx (int), kwarg Tensors keyed by name (str). Skip non-Tensors in both.
> Keywords: parents, kwargs, mixed-keys, string-keyed
> ```

**KCs targeted:** `parents-dict-by-argidx`, `parents-kwarg-keyed-by-name`

Implement `build_parents_full(args, kwargs) -> dict`.

Return a single dict mixing two key types:
1. **Positional Tensors** → keyed by their original index (int). Skip non-Tensor positions.
2. **Kwarg Tensors** → keyed by their kwarg name (str). Skip non-Tensor kwarg values.

**Worked examples.**
```python
build_parents_full((t1, 3.0), {})            # {0: t1}
build_parents_full((), {'mask': t1})         # {'mask': t1}
build_parents_full((t1,), {'mask': t2})      # {0: t1, 'mask': t2}
build_parents_full((3.0, t1), {'a': 5, 'b': t2})   # {1: t1, 'b': t2}
```

The dict mixes int and str keys — Python permits this. The reverse pass looks up positional back fns by int and kwarg back fns by string, so the key type DOES carry information.

Use `MiniTensor` (defined in the preamble) as the Tensor type to test against.

In [ ]:
def build_parents_full(args, kwargs):
    parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
    parents.update(
        {k: v for k, v in kwargs.items() if isinstance(v, MiniTensor)}
    )
    return parents


<details><summary>Solution</summary>

```python
def build_parents_full(args, kwargs):
    parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
    parents.update(
        {k: v for k, v in kwargs.items() if isinstance(v, MiniTensor)}
    )
    return parents
```

**Why mix key types in one dict.** Python dicts are heterogeneous-key by design. The reverse pass distinguishes positional from kwarg back fns by the key TYPE: `isinstance(k, int)` → positional back fn at argnum `k`; `isinstance(k, str)` → kwarg back fn for kwarg name `k`. One unified container, two lookup paths.

**Why kwargs Tensors need a back fn at all.** Some torch ops take tensor-valued kwargs that are differentiable (e.g. an `index_select` with a `weights=tensor` kwarg, or attention with a `mask` that participates in the gradient). Treating them as parents lets the reverse pass route gradients back to them.

**The `'0'` vs `0` test catches a subtle collapse bug.** A naive implementation might `str(i)` the positional keys to unify the dict; that would clash if a user ever passed a kwarg literally named `'0'`. Keeping int keys as int avoids the collision.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()